In [1]:
import json
from collections import defaultdict
import xml.etree.ElementTree as ET
import pandas as pd
import numpy as np
from mlx_lm import load, generate
from tqdm.notebook import tqdm

In [2]:
with open("./v1.5/dev/archehr-qa_key.json") as f:
    key = json.load(f)


tree = ET.parse("./v1.5/dev/archehr-qa.xml")
root = tree.getroot()

data = []
for case in root.findall("case"):
    case_id = case.get('id')
    c_question = case.findtext("clinician_question", default="").strip()
    p_question = case.findtext("patient_narrative", default="").strip()
    sentences = [
        {
            'sentence_id': sentence.get('id'),
            'text': sentence.text.strip()
        }
        for sentence in case.find("note_excerpt_sentences")
    ]
    item = {
        'case_id': case_id,
        'excerpt_sentences': sentences,
        'clinician_question': c_question,
        'patient_question': p_question,
    }
    data.append(item)

df1 = pd.DataFrame(data)
df2 = pd.DataFrame(key)
combined = df1.merge(df2[['case_id', 'clinician_answer_sentences']], on='case_id').to_dict(orient='records')

In [3]:
model, tokenizer = load("RepublicOfKorokke/Qwen3.5-35B-A3B-mlx-lm-mxfp4")

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

In [4]:
SYSTEM_PROMPT_1 = """You are a clinical evidence verification system.

Task:
Determine whether a clinical note sentence directly supports an answer sentence.

Rules:
- Answer YES or NO only.
- YES means the note sentence directly states or clearly entails the answer sentence.
- NO means it does not.
- Do not explain.
- Output only YES or NO."""


USER_PROMPT_1 = """Answer Sentence:
{}

Clinical Note Sentence:
{}

Does the clinical note sentence directly support the answer sentence?"""

from mlx_lm.sample_utils import make_sampler

sampler = make_sampler(temp=0.0)

def stage_1(*args):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT_1},
        {"role": "user", "content": USER_PROMPT_1.format(*args)},
    ]
    prompt = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, enable_thinking=False
    )
    
    text = generate(model, tokenizer, prompt=prompt, max_tokens=10, sampler=sampler)

    return text

In [6]:
from tqdm.notebook import tqdm
from ast import literal_eval

dev_answers = []
for case in tqdm(combined):
    cid = case['case_id']
    item = {'case_id': cid, 'prediction': []}
    
    for ans_sent in case['clinician_answer_sentences']:
        pred = {'answer_id': ans_sent['id'], 'evidence_id': []}
        
        for ev_sent in case['excerpt_sentences']:
            response = stage_1(ans_sent['text'], ev_sent['text']).strip()
            
            if response == 'YES':
                pred['evidence_id'].append(ev_sent['sentence_id'])
            elif response == 'NO':
                continue
            else:
                print(f"Unexpected response for case {cid}, answer{ans_sent['id']}, evidence {ev_sent['sentence_id']}")

        item['prediction'].append(pred)   
    
    dev_answers.append(item)

  0%|          | 0/20 [00:00<?, ?it/s]

In [7]:
from scoring_subtask_4 import compute_alignment_scores, get_leaderboard, load_key

key_map = load_key("./v1.5/dev/archehr-qa_key.json")

scores = compute_alignment_scores(dev_answers, key_map)
leaderboard = get_leaderboard(scores)
leaderboard

{'micro_precision': 35.35353535353536,
 'micro_recall': 15.837104072398189,
 'micro_f1': 21.875,
 'macro_precision': np.float64(38.84325396825397),
 'macro_recall': np.float64(22.664232896969473),
 'macro_f1': np.float64(25.261172436095038),
 'overall_score': 21.875}